In [1]:
import sklearn
from sklearn import svm
import pandas as pd
import numpy as np
import spacy
import nltk
from nltk.util import ngrams
import os
import zipfile

Importo i csv

In [2]:
train_path='../data/desegma-it.subTaskA.shared.train.0923-1220.csv'
test_path='../data/desegma-it.subTaskA.shared.test.1117_1835.csv'
test_with_labels_path='../data/desegma-it.subTaskA.with_labels.test.1117_1835.csv'

Creo un training set bilanciato

In [3]:
# 1. Carica il file di training originale
df_full = pd.read_csv(train_path)

# 2. Separa le due classi in base a 'label'
df_class_0 = df_full[df_full['label'] == 0]
df_class_1 = df_full[df_full['label'] == 1]

# 3. Estrai 1000 campioni casuali da ciascuna
train_0 = df_class_0.sample(n=1000, random_state=42)
train_1 = df_class_1.sample(n=1000, random_state=42)

# 4. Salva gli indici ORIGINALI prima di qualsiasi reset
train_indices = pd.concat([train_0, train_1]).index

# 5. Uniscili, mischia e resetta l'indice
df_train_final = pd.concat([train_0, train_1]).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Training set creato: {len(df_train_final)} documenti.")

Training set creato: 2000 documenti.


Creo un validation set assicurandomi che i testi siano diversi da quelli del training

In [4]:
# --- A. ESTRAZIONE VALIDATION (dal residuo del training) ---

# Escludi dal file originale gli indici ORIGINALI usati per il training
df_remaining = df_full.drop(train_indices)

# Separa per classi nel residuo
df_rem_0 = df_remaining[df_remaining['label'] == 0]
df_rem_1 = df_remaining[df_remaining['label'] == 1]

# Estrai 500+500 per il validation set
val_0 = df_rem_0.sample(n=500, random_state=42)
val_1 = df_rem_1.sample(n=500, random_state=42)
df_val_final = pd.concat([val_0, val_1]).sample(frac=1, random_state=42).reset_index(drop=True)

# --- B. ESTRAZIONE TEST (dal file di test separato) ---

df_test_full = pd.read_csv(test_with_labels_path)

# Separa le classi nel file di test
df_test_0 = df_test_full[df_test_full['label'] == 0]
df_test_1 = df_test_full[df_test_full['label'] == 1]

# Estrai 500+500 per il test set
test_0 = df_test_0.sample(n=500, random_state=42)
test_1 = df_test_1.sample(n=500, random_state=42)
df_test_final = pd.concat([test_0, test_1]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Training set:   {len(df_train_final)} doc (1000 class 0, 1000 class 1)")
print(f"Validation set: {len(df_val_final)} doc  (500 class 0,  500 class 1)")
print(f"Test set:       {len(df_test_final)} doc  (500 class 0,  500 class 1)")

# Verifica che non ci siano sovrapposizioni tra train e val
overlap = set(train_indices) & set(pd.concat([val_0, val_1]).index)
print(f"Sovrapposizioni train/val: {len(overlap)}")  # deve essere 0

Training set:   2000 doc (1000 class 0, 1000 class 1)
Validation set: 1000 doc  (500 class 0,  500 class 1)
Test set:       1000 doc  (500 class 0,  500 class 1)
Sovrapposizioni train/val: 0


Data Preparation: estrazione di token, pos e lemmi

In [5]:
#!python -m spacy download it_core_news_sm
nlp = spacy.load("it_core_news_sm", disable=["parser", "ner"])
doc = nlp("Apple is looking at buying U.K. startup for $1 billion")
for token in doc:
    print(token.text, token.pos_, token.lemma_)

Apple PROPN Apple
is VERB isre
looking X looking
at PROPN at
buying NOUN buying
U.K. PROPN U.K.
startup VERB startup
for ADP for
$ NOUN $
1 NUM 1
billion NOUN billion


In [6]:
print(nlp.pipe_names)

['tok2vec', 'morphologizer', 'tagger', 'lemmatizer', 'attribute_ruler']


In [7]:
df_train_final.head()

,text,label
0,"Eppure, nel mezzo di queste scorse commerciali...",1
1,"""È una cosa che sto valutando di fare da fine ...",0
2,Il caso Pirelli si presenta non come un sempli...,1
3,Poi la direzione distrettuale antimafia di Lec...,0
4,Il cambio di programma si è configurato in una...,1


In [8]:
# Carichiamo il modello una sola volta all'inizio dello script
nlp = spacy.load("it_core_news_sm")

def preprocess_dataframe(df, text_column='text'):
    """
    Esegue il preprocessing NLP su una colonna di un DataFrame.
    Aggiunge le colonne: tokens, pos, lemmas, tokens_processed, pos_processed, lemmas_processed.
    """
    # Liste temporanee
    tokens_list = []
    pos_list = []
    lemmas_list = []

    print(f"Elaborazione in corso per {len(df)} righe...")

    # Batch processing con nlp.pipe
    for doc in nlp.pipe(df[text_column], disable=["ner", "parser"]):
        
        ## 1. Estrazione dati base (filtrando punteggiatura)
        #t_row = [token.text.lower() for token in doc if not token.is_punct]
        #p_row = [token.pos_ for token in doc if not token.is_punct]
        #l_row = [token.lemma_.lower() for token in doc if not token.is_punct]

        # 1. Estrazione dati base
        t_row = [token.text.lower() for token in doc]
        p_row = [token.pos_ for token in doc]
        l_row = [token.lemma_.lower() for token in doc]
        
        # Append alle liste
        tokens_list.append(t_row)
        pos_list.append(p_row)
        lemmas_list.append(l_row)

    # 3. Assegnazione multipla al DataFrame originale
    # Usiamo .assign o l'assegnazione diretta. Per chiarezza:
    df['tokens'] = tokens_list
    df['pos'] = pos_list
    df['lemmas'] = lemmas_list
    
    # 4. Creazione versione stringa per TF-IDF
    df['tokens_processed'] = df['tokens'].apply(lambda x: " ".join(x))
    df['pos_processed'] = df['pos'].apply(lambda x: " ".join(x))
    df['lemmas_processed'] = df['lemmas'].apply(lambda x: " ".join(x))
    
    print("Completato!")
    return df

In [9]:
df_train_final = preprocess_dataframe(df_train_final)
df_test_final = preprocess_dataframe(df_test_final)
df_val_final = preprocess_dataframe(df_val_final)

Elaborazione in corso per 2000 righe...
Completato!
Elaborazione in corso per 1000 righe...
Completato!
Elaborazione in corso per 1000 righe...
Completato!


Salvo i 3 df preprocessati in formato pickle

In [ ]:
df_train_final.to_pickle("../data/processed/df_train_processed.pkl")
df_val_final.to_pickle("../data/processed/df_val_processed.pkl")
df_test_final.to_pickle("../data/processed/df_test_processed.pkl")

## 2. Esportazione Unificata dei Dataset per Profiling-UD

Per garantire la perfetta **coerenza dimensionale** delle feature linguistiche ed evitare asimmetrie strutturali tra le matrici di addestramento e di test, si adotta una strategia di **esportazione unificata**. 

Il tool *Profiling-UD* genera le colonne dell'output in modo dinamico, basandosi esclusivamente sulle strutture morfosintattiche che incontra durante l'analisi. Elaborare i tre split separatamente comporterebbe il rischio quasi certo di ottenere un numero di feature differente tra Train, Validation e Test (a causa di costrutti rari presenti in un set ma assenti negli altri), causando errori di sfasamento dimensionale all'interno della SVM lineare.

Per superare questo limite tecnico, il blocco di codice seguente implementa i seguenti passaggi:
* **Directory di output centralizzata:** Viene configurata un'unica cartella temporanea in cui far confluire tutti i testi.
* **Marcatura tramite prefissi univoci:** Ciascun documento viene salvato come file `.txt` utilizzando una convenzione di denominazione rigida basata sul proprio split di provenienza (`train_`, `val_`, `test_`) unito all'indice numerico del dataframe originario. Questo approccio è cruciale per poter riallineare gli ID e ri-separare chirurgicamente le righe in un secondo momento, azzerando il rischio di *data leakage*.
* **Generazione del pacchetto ZIP:** Tutti i file così generati vengono compressi in un unico archivio `.zip`, pronto per essere inviato in blocco al sistema di profilazione linguistica per l'estrazione di un unico schema di feature condiviso.

In [11]:
# 1. Definisci un'unica cartella di output
outdir = '../data/processed/data_task2/all_data_profiling'
os.makedirs(outdir, exist_ok=True)

# 2. Salva i file di TRAIN
for i, text in enumerate(df_train_final['text']):
    file_path = os.path.join(outdir, f'train_{i}.txt')
    with open(file_path, "w", encoding='utf-8') as f:
        f.write(str(text))

# 3. Salva i file di TEST
for i, text in enumerate(df_test_final['text']):
    file_path = os.path.join(outdir, f'test_{i}.txt')
    with open(file_path, "w", encoding='utf-8') as f:
        f.write(str(text))

# 4. Salva i file di VALIDATION
for i, text in enumerate(df_val_final['text']):
    file_path = os.path.join(outdir, f'val_{i}.txt')
    with open(file_path, "w", encoding='utf-8') as f:
        f.write(str(text))

# 5. Crea un unico file ZIP contenente tutti i documenti
zip_path = '../data/processed/data_task2/all_data_profiling.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for fname in os.listdir(outdir):
        zipf.write(os.path.join(outdir, fname), arcname=fname)

print(f"Tutti i documenti sono stati zippati con successo in: {zip_path}")

Tutti i documenti sono stati zippati con successo in: ../data/processed/data_task2/all_data_profiling.zip
